In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path
import os
import re

This is the data from the email! Had a lot of bad luck with trying to get it myself... (see doc, when written).
Nothng is particularly novel yet. And sorry, no pipeline... (but a few steps are very general).

In [2]:
data_path = os.path.abspath("../data/tcga_data2.csv")

raw_df = pd.read_table(data_path, sep = '\t', low_memory = False)
raw_df.head(1)

,Gene,Study of Origin,Sample ID,Cancer Type,Cancer Type Detailed,Protein Change,Annotation,Custom Driver,Custom Driver Tiers,Functional Impact,...,Tumor Type,Used in Genomic Analysis,Vascular invasion indicator,Vessel Invasion,Vial number,Patient's Vital Status,Patient Weight,WGD,Winter Hypoxia Score,Year of Diagnosis
0,APC,"Colorectal Adenocarcinoma (TCGA, Firehose Legacy)",TCGA-AA-A010-01,Colorectal Cancer,Colon Adenocarcinoma,A2D,"OncoKB: Unknown, level NA, resistance NA;reVUE...",NaN,NaN,MutationAssessor: NA;SIFT: impact: deleterious...,...,NaN,NaN,NO,NaN,A,NaN,NaN,NaN,NaN,NaN


In [3]:
# https://docs.python.org/3/library/re.html
# \b is a space
cols = np.array(raw_df.columns.tolist())
pattern = re.compile(r'\bage\b', flags = re.IGNORECASE)
age_cols = np.array([])

for col in cols:
    # Regex needs pattern.search, instead of say "str".in(array)
    if pattern.search(col):
        age_cols = np.append(age_cols, col)

age_cols

array(['Diagnosis Age', 'Age at derivation', 'Age at Diagnosis',
       'Age At Diagnosis', 'Age at Metastases',
       'Age at Which Sequencing was Reported (Days)',
       'Age at Which Sequencing was Reported (Years)'], dtype='<U44')

In [4]:
note_cols = ["Gene", "Sample ID", "Cancer Type Detailed", "Mutation Type", "Variant Type", "HGVSc", "MS", 
        "Protein Change", "Functional Impact"]

good_cols = np.concatenate((note_cols, age_cols))
df = raw_df[good_cols].copy()

# Gene expression as in email
# df = short_df[short_df["HGVSc"] == "ENST00000257430.4:c.835-8A>G"]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6501 entries, 0 to 6500
Data columns (total 16 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Gene                                          6501 non-null   object 
 1   Sample ID                                     6501 non-null   object 
 2   Cancer Type Detailed                          6501 non-null   object 
 3   Mutation Type                                 6501 non-null   object 
 4   Variant Type                                  6498 non-null   object 
 5   HGVSc                                         5660 non-null   object 
 6   MS                                            3188 non-null   object 
 7   Protein Change                                6501 non-null   object 
 8   Functional Impact                             6501 non-null   object 
 9   Diagnosis Age                                 3096 non-null   f

In [5]:
3096 + 70 + 1477 + 304 + 502 + 332 + 7

5788

In [ ]:
# Stolen bfill from you! Axis = 1 looks across columns
df.loc[:, 'Age'] = df[age_cols].bfill(axis = 1).iloc[:, 0]
df = df.drop(columns = age_cols)

In [7]:
early_age = 50
df["Early Onset"] = df["Age"] < early_age
df.head(1)

,Gene,Sample ID,Cancer Type Detailed,Mutation Type,Variant Type,HGVSc,MS,Protein Change,Functional Impact,Age,Early Onset
0,APC,TCGA-AA-A010-01,Colon Adenocarcinoma,Missense_Mutation,SNP,ENST00000257430.4:c.5C>A,Somatic,A2D,MutationAssessor: NA;SIFT: impact: deleterious...,46.0,True


---

---

Next steps could include:
1. Remove any entries with no age data,
1. Start some Bayesian stats to test whether c.835-8A>G carriers are younger at diagnosis than patients with other APC mutations, (some simple models would be easy enough to run in R),
1. Look at the mutation type `value_counts()` and see if the mutation type differs across age groups,
1. Visualise both of these ideas.